# Phase B: Engram with GPT-4o in place of Gemma

Two tests, both with **GPT-4o** (`gpt-4o-2024-08-06`) writing, on `manjunathhari/erc` main (ERC-4.0 General Core). ERC is not used yet: this sets the GPT-4o baseline.

1. **LoCoMo final test**, conversations 5 to 9, 776 questions, the same design and prompts as the Gemma final test. GPT-4o writes the noted facts and the answers. It is compared three ways: with facts, without facts, and the older search.
2. **LongMemEval**, all 470 test questions, with the official GPT-4o judge.

**Before running:**
- Add your key under *Add-ons > Secrets* as `OPENAI_API_KEY`, attached to this notebook. Never paste it into a cell.
- *Settings > Internet* on. No GPU needed.

**To run unattended:** use *Save Version > Save & Run All (Commit)*. It keeps running with the tab closed, up to 12 hours; expect about 4 hours. Both tests resume where they stopped if run again.

**Cost:** roughly $10–15 of GPT-4o usage in total. This is an estimate.

In [ ]:
COMMIT = "99f34f0"   # manjunathhari/erc, phase B
import os, subprocess, shutil
WORK = "/kaggle/working"
os.chdir(WORK)
subprocess.run("rm -rf erc; git clone -q https://github.com/manjunathhari/erc.git", shell=True, check=True)
os.chdir(f"{WORK}/erc")
subprocess.run(f"git checkout -q {COMMIT}; git clone -q --depth 1 https://github.com/snap-research/locomo.git", shell=True, check=True)
subprocess.run("pip install -q wordllama scikit-learn spacy openai; python -m spacy download en_core_web_sm", shell=True, check=True, capture_output=True)
import wordllama
p = os.path.dirname(wordllama.__file__)
a, b = os.path.join(p, "tokenizers"), os.path.join(p, "tokenizer")
if os.path.isdir(a) and not os.path.exists(b): shutil.copytree(a, b)
DATA = f"{WORK}/longmemeval_s_cleaned.json"
if not os.path.exists(DATA):
    subprocess.run(f"wget -q -O {DATA} https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json", shell=True, check=True)
print("LongMemEval data:", os.path.getsize(DATA) // 1000000, "MB")

from kaggle_secrets import UserSecretsClient
os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY").strip()
from openai import OpenAI
print("key works, GPT-4o visible:", "gpt-4o-2024-08-06" in {m.id for m in OpenAI().models.list()})

import sys; sys.path.insert(0, os.getcwd())
from engram_memory import open_memory
m = open_memory("/tmp/check.db", durable=False); print("tested design loads:", type(m.search).__name__, m.search.mode)
print("setup ready, commit", subprocess.run("git log --oneline -1", shell=True, capture_output=True, text=True).stdout.strip())

In [ ]:
# Test 1: LoCoMo final test with GPT-4o (about 2 hours)
SAVE = f"{WORK}/engram_results"
!cd {WORK}/erc && python -u kit_tools/locomo_gpt4o.py --save {SAVE} 2>&1 | tail -70

In [ ]:
# Test 2: LongMemEval, all 470 test questions (about 2 hours)
!cd {WORK}/erc && python -u kit_tools/lme_gpt4o.py --data {DATA} --save {SAVE} --n 0 2>&1 | tail -50

In [ ]:
print(sorted(os.listdir(SAVE)))

When both say **Done**, download these from the `engram_results` folder in the Output panel and send them to Claude:

- `locomo_gpt4o_summary.json`, `locomo_gpt4o_answers.jsonl`, `locomo_gpt4o_facts.json`
- `lme_gpt4o_summary.json`, `lme_gpt4o_answers.jsonl`

Claude then adds the blind strict and generous labels on LoCoMo, checks a sample of the GPT-4o facts against their sentences, and compares with the Gemma results.